# Patchscopes: Representation Inspection

Patchscopes (Ghandeharioun et al., ICML 2024) is a unifying framework for
inspecting hidden representations: patch an activation from one context into
a second "inspection" context and let the model decode what it represents.

This notebook covers the `irtk.patchscopes` module.

## Why This Matters

Patchscopes inspect what information is encoded at specific positions by patching activations into different contexts. This extends the logit lens idea: instead of just projecting through the unembedding, you can test what a representation 'means' by placing it in a context designed to extract that meaning.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import patchscopes

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Core Patchscope

Patch a source activation into a target context.

In [ ]:
source = model.to_tokens("The Eiffel Tower is located in")
target = model.to_tokens("The answer is")

# Patch layer 8 residual from source into the last position of target
patched_logits = patchscopes.patchscope(
    model, source, "blocks.8.hook_resid_post",
    target, "blocks.11.hook_resid_post", target_pos=-1
)

# What does the model predict?
probs = jax.nn.softmax(jnp.array(patched_logits[-1]))
top_idx = np.argsort(np.array(probs))[::-1][:5]
print("Top predictions after patching:")
for idx in top_idx:
    print(f"  {tokenizer.decode([idx]):>15s}: {float(probs[idx]):.4f}")

## 2. Logit Lens as Patchscope

The standard logit lens is a special case of Patchscopes.

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")

# Compare logit lens via patchscope at each layer
print("Logit lens patchscope at each layer (last position):")
for layer in range(0, model.cfg.n_layers, 3):
    logits = patchscopes.logit_lens_patchscope(model, tokens, layer, pos=-1)
    top_idx = int(np.argmax(logits))
    print(f"  Layer {layer:2d}: top prediction = '{tokenizer.decode([top_idx])}' "
          f"(logit={logits[top_idx]:.3f})")

## 3. Token Identity Inspection

What entity has the model resolved at a given position and layer?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")

# Inspect the last position at several layers
for layer in [0, 4, 8, 11]:
    result = patchscopes.token_identity_inspection(model, tokens, layer, pos=-1, k=5)
    top = result['top_tokens'][:3]
    top_str = ', '.join([f"'{tokenizer.decode([tid])}' ({p:.3f})" for tid, p in top])
    print(f"Layer {layer:2d}: H={result['entropy']:.2f}  top: {top_str}")

## 4. Attribute to Source

Sweep over (layer, position) pairs to find what drives a target prediction.

In [ ]:
source = model.to_tokens("The Eiffel Tower is located in")
target = model.to_tokens("The answer is")
paris_id = tokenizer.encode(" Paris")[0]

def metric(logits):
    return float(logits[-1, paris_id])

result = patchscopes.attribute_to_source(
    model, source, target,
    "blocks.11.hook_resid_post", -1, metric,
    layers=list(range(0, 12, 2))
)

fig, ax = plt.subplots(figsize=(12, 4))
token_strs = [tokenizer.decode([int(t)]) for t in source]
im = ax.imshow(result['attribution_matrix'], cmap='RdBu_r', aspect='auto')
ax.set_yticks(range(len(result['attribution_matrix'])))
ax.set_yticklabels([f'Layer {i}' for i in range(0, 12, 2)])
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha='right')
ax.set_title('Source Attribution for "Paris" prediction')
plt.colorbar(im, ax=ax, label='Metric value')
plt.tight_layout()
plt.show()

print(f"Best source: layer {result['best_layer']}, pos {result['best_pos']}")

## 5. Cross-Model Inspection

Use one model to decode another model's representations.

In [ ]:
# Using the same model as both source and target for demonstration
# In practice, you'd use different model sizes
source_tokens = model.to_tokens("The capital of France is")
target_tokens = model.to_tokens("The answer is")

result = patchscopes.cross_model_inspection(
    model, model,
    source_tokens, "blocks.8.hook_resid_post",
    target_tokens, "blocks.11.hook_resid_post",
    target_pos=-1
)

probs = np.array(jax.nn.softmax(jnp.array(result[-1])))
top_idx = np.argsort(probs)[::-1][:5]
print("Cross-model inspection predictions:")
for idx in top_idx:
    print(f"  {tokenizer.decode([idx]):>15s}: {probs[idx]:.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `patchscope()` | Core: patch source activation into target context |
| `logit_lens_patchscope()` | Logit lens as a Patchscope special case |
| `token_identity_inspection()` | Probe resolved token identity at layer/position |
| `attribute_to_source()` | Sweep (layer, pos) to find what drives a metric |
| `cross_model_inspection()` | Patch between different models |